## 4.3 Pytorch 搭建MLP - Sequential 写法

在上一节中，我们使用的是 自定义 nn.Module 的方式 来构建 MLP。

这种方式是 最通用、最灵活 的方法。

但在很多简单的神经网络中，如果网络结构只是：

`Layer → Activation → Layer → Activation`

这种 顺序结构（Sequential Structure），PyTorch 提供了一种更简单的写法：

`nn.Sequential`

#### 1. 什么是 Sequential？
Sequential 的意思是：

`按照顺序依次执行多个层（layer）。`

也就是说，数据会按照我们定义的顺序：

`输入 → 第一层 → 第二层 → 第三层 → 输出`

自动依次执行。

##### 1.1 Sequential 的核心思想
如果网络结构是：

`x → Linear → ReLU → Linear → Output`

我们可以直接写成：
```
nn.Sequential(
    layer1,
    layer2,
    layer3
)
```
Sequential 会自动执行：
1. layer1(x)
2. layer2(...)
3. layer3(...)

#### 2. 使用 Sequential 构建 MLP

##### 2.1 使用 Sequential 定义模型

In [1]:
import torch 
import torch.nn as nn

model = nn.Sequential(
    nn.Linear(3, 4),
    nn.ReLU(),
    nn.Linear(4,1)
)

##### 2.2 网络结构解释
这个模型对应的网络结构是：
```
Input(3)
   ↓
Linear(3 → 4)
   ↓
ReLU
   ↓
Linear(4 → 1)
   ↓
Output
```

##### 2.3 前向传播自动完成
如果我们输入数据：
`X = torch.randn(5, 3)`

执行：
`y_pred = model(X)`

Sequential 会自动执行：
```
X
↓
Linear(3→4)
↓
ReLU
↓
Linear(4→1)
↓
Output
```
我们不需要自己写 forward()。

#### 3. Sequential 的内部原理
Sequential 本质上也是一个：

`nn.Module`

只不过它内部自动帮我们实现了：

`forward()`

可以这样理解：

`Sequential = --init()__ + 自动写好的 forward()`

例如：
``` python
def forward(x):

    x = self.layer1(x)
    x = self.layer2(x)
    x = selflayer3(x)

    return x
```
所以：
`model(X)`

仍然会自动调用：
`forward()`

#### 4. Sequential 的另一种写法（带名字）
有时我们希望给每一层一个名字。
可以使用：

In [2]:
from collections import OrderedDict

model_named = nn.Sequential(
    OrderedDict([
        ("fc1", nn.Linear(3, 4)),
        ("relu1", nn.ReLU()),
        ("fc2", nn.Linear(4,1))
    ])
)

这样打印模型时会显示：

In [3]:
print(model_named)

Sequential(
  (fc1): Linear(in_features=3, out_features=4, bias=True)
  (relu1): ReLU()
  (fc2): Linear(in_features=4, out_features=1, bias=True)
)


#### 5. Sequential vs 自定义 Module

##### 5.1 自定义 Module 写法
``` python
class MLP(nn.Module):

    def __init__(self):
        super().__init__()

        self.layer1 = nn.Linear(3,4)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(4,1)

    def forward(self,x):

        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)

        return x
```
特点：
* 灵活
* 可以写复杂结构
* 可以有分支结构
* 可以写循环
* 可以做残差连接

##### 5.2  Sequential 写法
``` python
model = nn.Sequential(

    nn.Linear(3,4),
    nn.ReLU(),
    nn.Linear(4,1)

)
```
特点：
* 写法非常简洁
* 不需要写 forward
* 适合简单结构

#### 6. 什么时候使用 Sequential？

##### 6.1 适合 Sequential 的情况
网络结构是 单一路径：

`Layer → Layer → Layer`

例如：
* 简单 MLP
* 简单 CNN

##### 6.2 不适合 Sequential 的情况
如果网络结构比较复杂，例如：
* ResNet（残差连接）
* 多输入网络
* 多输出网络
* Attention 结构
* Transformer

这种情况下必须使用：

自定义 nn.Module